In [1]:
%load_ext autoreload
%autoreload 2

from dataclasses import dataclass
import numpy as np
import torch

from gtfm.viz.graph import draw_scms

from tfmplayground.priors.dataloader import TabICLPriorDataLoader
from tfmplayground.priors.utils import dump_prior_to_h5
from tfmplayground.utils import get_default_device

from tabicl.prior.prior_config import DEFAULT_FIXED_HP, DEFAULT_SAMPLED_HP
from tabicl.prior.dataset import PriorDataset as TabICLPriorDataset

# TFM-Playground

In [2]:
@dataclass
class PriorConfig:
    num_batches: int = 1000
    batch_size: int = 8
    min_features: int = 5
    max_features: int = 5
    min_seq_len: int | None = None
    max_seq_len: int = 50
    max_classes: int = 3
    save_path: str = '../data/tabicl_4k_50x3.h5'
    device: str = get_default_device()

cfg = PriorConfig()

In [3]:
# Custom SCM prior hyperparameters
cfg.scm_fixed_hp = {
    **DEFAULT_FIXED_HP,
    'is_causal': True,
    'in_clique': False,
    # 'block_wise_dropout': False,
    # 'mlp_dropout_prob': 0.5,
    # 'num_layers': 4,
    'hidden_dim': 3,
    # 'num_causes': 2,
}
cfg.scm_sampled_hp = {k: v for k, v in DEFAULT_SAMPLED_HP.items() if k not in cfg.scm_fixed_hp}

cfg.min_features, cfg.max_features = 10, 10

In [4]:
prior = TabICLPriorDataLoader(
    num_steps=cfg.num_batches,
    batch_size=cfg.batch_size,
    num_datapoints_min=cfg.min_seq_len,
    num_datapoints_max=cfg.max_seq_len,
    min_features=cfg.min_features,
    max_features=cfg.max_features,
    max_num_classes=cfg.max_classes,
    device=cfg.device,

    scm_fixed_hp=cfg.scm_fixed_hp,
    scm_sampled_hp=cfg.scm_sampled_hp,
)
problem_type = "classification"

In [5]:
batch = next(iter(prior))
print(batch.keys())
{k: (v.shape if isinstance(v, torch.Tensor) else type(v)) for k, v in batch.items()}

dict_keys(['x', 'y', 'target_y', 'single_eval_pos', 'adj', 'priors'])


{'x': torch.Size([8, 50, 10]),
 'y': torch.Size([8, 50]),
 'target_y': torch.Size([8, 50]),
 'single_eval_pos': int,
 'adj': torch.Size([8, 11, 11]),
 'priors': tuple}

In [6]:
# batch = xs, ys, active_featureses, seqlens, train_sizes, scm_prios = next(prior.pd)
# # print(xs.shape, ys.shape, active_featureses.shape, seqlens.shape, train_sizes.shape, len(scm_prios))
# # pd_idx = 1
# # x, y, active_features, seqlen, train_size, scm_prior = xs[pd_idx], ys[pd_idx], active_featureses[pd_idx], seqlens[pd_idx], train_sizes[pd_idx], scm_prios[pd_idx]

# assert (active_featureses[0] == active_featureses[1:]).all()
# graphs_full = []
# graphs_moma = [] #moma = moralized, marginalized
# nodelists = []
# for x, y, active_features, seqlen, train_size, scm_prior in zip(*batch):
#     assert (x[:, active_features.item():] == 0.).all()
#     graphs_full.append(scm_prior.graph)
#     graphs_moma.append(scm_prior.graph_moral_marg)
#     nodelists.append(scm_prior.graph.graph['nodes_include'])

# n_graphs = len(graphs_full)
# n_cols = 8

# draw_scm_params = dict(
#     with_labels = True,
#     n_rows = np.ceil(n_graphs / n_cols).astype(int),
#     figsize = (3, 4),
# )

# draw_scms(
#     graphs_full, 
#     suptitle='Original graph',
#     edge_alpha = 0.3,
#     **draw_scm_params,
# )


# draw_scms(
#     graphs_moma, 
#     suptitle='Moralized and marginalized graph',
#     nodelists= nodelists,
#     edge_alpha = 1.,
#     **draw_scm_params,
# )


# # draw_scms(
# #     [scm_prior.graph, scm_prior.graph_moral_marg],
# #     with_labels = True,
# #     edge_alpha = 1.,
# #     figsize = (3, 4),
# # )

In [7]:
dump_prior_to_h5(prior, cfg.max_classes, cfg.batch_size, cfg.save_path, problem_type, cfg.max_seq_len, cfg.max_features)